#Configuração do Ambiente de IA Generativa e RAG

Este bloco de comandos instala as bibliotecas necessárias para construir aplicações de Inteligência Artificial, RAG (Recuperação de Informação) e agentes autônomos no Google Colab.

In [1]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requir

### 📚 Explicação detalhada dos pacotes

| Pacote | Função Principal | Para que serve no projeto? |
| :--- | :--- | :--- |
| **`pypdf`** | Leitura e extração de PDF | Carrega e extrai texto de documentos PDF para serem processados pela IA (ideal para RAG). |
| **`langchain`** | Framework principal de IA | Orquestra as chamadas para os modelos, gerencia prompts e cria cadeias de processamento. |
| **`langchain-community`** | Integrações da comunidade | Conecta o projeto a bancos vetoriais, ferramentas externas e leitores de dados variados. |
| **`langchain-groq`** | Conector do provedor Groq | Permite usar a API da Groq para rodar modelos abertos (como *Llama 3*) em altíssima velocidade. |
| **`langchain-huggingface`** | Modelos e Embeddings | Conecta a modelos de linguagem e de *embeddings* (vetores de texto) hospedados no Hugging Face. |
| **`langgraph`** | Agentes autônomos e Grafos | Cria fluxos avançados de IA com loops de decisão, persistência de estado e múltiplos agentes. |

---

#### 💡 O que significam as flags do `%pip`?
* **`-U`** (*Upgrade*): Garante que a versão mais recente e atualizada do pacote será instalada.
* **`-q`** (*Quiet*): Oculta mensagens longas de instalação, mantendo a saída da célula limpa e organizada.

# Documento PDF da LinkedIn Community

In [2]:
url = 'https://raw.githubusercontent.com/allanspadini/curso-flash-rag/main/m2m_strategy_and_objectives_development.pdf'

# Utilizando o PyPDFLoader

**- Importação do PyPDFLoader<br>**
**- Utilizamos o PyPDFLoader para criar uma variável chamada loader, que será igual a PyPDFLoader(url), definindo assim nosso loader:<br>**
**- No entanto, não queremos carregar todo o documento em um único bloco; vamos carregá-lo dividido em páginas, por exemplo. Para isso, criamos uma lista vazia chamada pages:<br>**
**- E utilizamos um laço for para iterar sobre loader.lazy_load(), adicionando página por página dentro dessa lista:**


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(url)
pages = []
for page in loader.lazy_load():
    pages.append(page)

/tmp/ipykernel_742/3712195572.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Explorando o conteúdo do documento
**- Para verificar a informação, podemos tentar imprimir o conteúdo da primeira página, que é a página zero. Podemos usar o comando:**

In [4]:
print(f"{pages[0].metadata}\n")

{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-04-06T17:28:28-04:00', 'moddate': '2023-04-06T17:29:25-04:00', 'trapped': '/False', 'source': 'https://raw.githubusercontent.com/allanspadini/curso-flash-rag/main/m2m_strategy_and_objectives_development.pdf', 'total_pages': 78, 'page': 0, 'page_label': '1'}



**- Visualizando o conteúdo da primeira página**

In [5]:
print(pages[0].page_content)

NASA’S 
MOON TO MARS 
STRATEGY AND 
OBJECTIVES 
DEVELOPMENT
A blueprint for sustained 
human presence and 
exploration throughout 
the Solar System
National Aeronautics and
Space Administration


# Criando uma base de dados vetorial
**- Para processar esses dados, vamos criar uma base de dados vetorial. O LangChain possui uma funcionalidade chamada InMemory VectorStore. Precisamos criar uma base de dados vetorial que chamaremos de VectorStore. Para isso, utilizamos o seguinte código:**

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

# Utilizando HuggingFace Embeddings
**- Vamos também utilizar o from LangChain_HuggingFace e entender o motivo disso. Podemos importar HuggingFace Embeddings. O que é o Embeddings? O processo de realizar o Embeddings transforma nosso texto em um formato numérico vetorial, facilitando buscas dentro desses documentos. Realizar uma busca em uma base de dados no formato numérico vetorial é muito mais eficiente. O processo de Embeddings transforma o texto nesse formato.
Para isso, utilizamos o seguinte código:**

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

**A OpenAI e o Google possuem suas próprias bibliotecas e modelos de IA que realizam esse processo de Embeddings. Aqui, utilizamos um modelo aberto disponível no HuggingFace, uma comunidade, site e biblioteca que oferece vários modelos de IA disponibilizados por empresas, instituições de pesquisa e pessoas físicas. Vamos utilizar um modelo interessante disponível na página do HuggingFace.**

In [8]:
embed_model = HuggingFaceEmbeddings(model_name="mixedbread-ai/mxbai-embed-large-v1")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

# Configurando o modelo de Embedding

In [9]:
vector_store = InMemoryVectorStore.from_documents(pages, embed_model)